# ETS Model Benchmark (Automatic Selection Only)

Parallel evaluation of ETS implementations with automatic component selection on M1, M3, and Tourism competition datasets.

**Packages evaluated:**
- **smooth** (ADAM, ES) - Automatic selection via Z (AIC-based)
- **statsforecast** - AutoETS (Nixtla)
- **sktime** - AutoETS
- **skforecast** - AutoETS

**Metrics:**
- RMSSE (Root Mean Squared Scaled Error)
- SAME (Scaled Absolute Mean Error)
- Computational time


In [1]:
import numpy as np
import pandas as pd
import time
from concurrent.futures import ProcessPoolExecutor, as_completed
import multiprocessing
import warnings
warnings.filterwarnings('ignore')

from fcompdata import M1, M3, Tourism

In [2]:
# Check which packages are available
from importlib.metadata import version as pkg_version
available_packages = {}

# Core environment (numpy/pandas affect every method's numeric backend)
import sys
print(f"python {sys.version.split()[0]}")
for _p in ('numpy', 'pandas'):
    print(f"{_p} v{pkg_version(_p)}")
print()

try:
    from smooth import ADAM, ES
    available_packages['smooth'] = True
    print(f"smooth v{pkg_version('smooth')}: Available")
except ImportError as e:
    available_packages['smooth'] = False
    print(f"smooth: Not available ({e})")

try:
    from statsforecast import models as sf_models
    available_packages['statsforecast'] = True
    print(f"statsforecast v{pkg_version('statsforecast')}: Available")
except ImportError as e:
    available_packages['statsforecast'] = False
    print(f"statsforecast: Not available ({e})")

try:
    from sktime.forecasting import ets as sktime_ets
    available_packages['sktime'] = True
    print(f"sktime v{pkg_version('sktime')}: Available")
except ImportError as e:
    available_packages['sktime'] = False
    print(f"sktime: Not available ({e})")

try:
    from skforecast.stats import Ets
    available_packages['skforecast'] = True
    print(f"skforecast v{pkg_version('skforecast')}: Available")
except ImportError as e:
    available_packages['skforecast'] = False
    print(f"skforecast: Not available ({e})")
try:
    from aeon.forecasting.stats import AutoETS as _AeonAutoETS
    available_packages['aeon'] = True
    print(f"aeon v{pkg_version('aeon')}: Available")
except ImportError as e:
    available_packages['aeon'] = False
    print(f"aeon: Not available ({e})")


python 3.14.4
numpy v2.5.1
pandas v3.0.3



smooth v1.0.7: Available
statsforecast: Not available (No module named 'statsforecast')
sktime: Not available (No module named 'sktime')
skforecast: Not available (No module named 'skforecast')
aeon: Not available (No module named 'aeon')


## Error Metrics

In [3]:
def RMSSE(holdout, forecast, actuals):
    """
    Root Mean Squared Scaled Error.
    """
    holdout = np.asarray(holdout)
    forecast = np.asarray(forecast)
    actuals = np.asarray(actuals)
    
    mse = np.mean((holdout - forecast) ** 2)
    scale = np.mean(np.diff(actuals) ** 2)
    
    if scale == 0:
        return np.nan
    
    return np.sqrt(mse / scale)

def SAME(holdout, forecast, actuals):
    """
    Scaled Absolute Mean Error.
    """
    holdout = np.asarray(holdout)
    forecast = np.asarray(forecast)
    actuals = np.asarray(actuals)
    
    ame = np.abs(np.mean(holdout - forecast))
    scale = np.mean(np.abs(np.diff(actuals)))
    
    if scale == 0:
        return np.nan
    
    return ame / scale

## Load Datasets

In [4]:
# Combine datasets into a list
datasets = []
for idx in M1.keys():
    datasets.append(M1[idx])
for idx in M3.keys():
    datasets.append(M3[idx])
for idx in Tourism.keys():
    datasets.append(Tourism[idx])

print(f"Total series: {len(datasets)}")
print(f"M1: {len(M1)} series")
print(f"M3: {len(M3)} series")
print(f"Tourism: {len(Tourism)} series")

Total series: 5315
M1: 1001 series
M3: 3003 series
Tourism: 1311 series


## Define Methods

Methods are defined as tuples: `(method_name, package, config_dict)`

In [5]:
# All method configurations (automatic selection only)
all_methods_config = [
    # smooth package - ADAM and ES models with automatic selection (Z = AIC-based)
    ("ADAM ETS Back", "smooth", {"class": "ADAM", "model": "ZXZ", "initial": "backcasting"}),
    ("ADAM ETS Opt", "smooth", {"class": "ADAM", "model": "ZXZ", "initial": "optimal"}),
    ("ADAM ETS Two", "smooth", {"class": "ADAM", "model": "ZXZ", "initial": "two-stage"}),
    ("ES Back", "smooth", {"class": "ES", "model": "ZXZ", "initial": "backcasting"}),
    ("ES Opt", "smooth", {"class": "ES", "model": "ZXZ", "initial": "optimal"}),
    ("ES Two", "smooth", {"class": "ES", "model": "ZXZ", "initial": "two-stage"}),
    ("ES XXX", "smooth", {"class": "ES", "model": "XXX", "initial": "backcasting"}),
    ("ES ZZZ", "smooth", {"class": "ES", "model": "ZZZ", "initial": "backcasting"}),
    ("ES FFF", "smooth", {"class": "ES", "model": "FFF", "initial": "backcasting"}),
    ("ES SXS", "smooth", {"class": "ES", "model": "SXS", "initial": "backcasting"}),

    # statsforecast - AutoETS (AIC-based selection)
    ("statsforecast AutoETS", "statsforecast", {}),

    # sktime - AutoETS (AIC-based selection)
    ("sktime AutoETS", "sktime", {}),

    # skforecast - Ets with automatic selection (ZZZ)
    ("skforecast AutoETS", "skforecast", {}),

    # aeon - AutoETS (AIC-based selection)
    ("aeon AutoETS", "aeon", {}),
]

# Filter to only available packages
methods_config = [
    (name, pkg, cfg) for name, pkg, cfg in all_methods_config 
    if available_packages.get(pkg, False)
]

methods_names = [m[0] for m in methods_config]
methods_number = len(methods_names)
dataset_length = len(datasets)

print(f"\nMethods available: {methods_number}")
print(f"Datasets: {dataset_length}")
print("\nMethod configurations:")
for name, pkg, cfg in methods_config:
    print(f"  {name} ({pkg})")


Methods available: 10
Datasets: 5315

Method configurations:
  ADAM ETS Back (smooth)
  ADAM ETS Opt (smooth)
  ADAM ETS Two (smooth)
  ES Back (smooth)
  ES Opt (smooth)
  ES Two (smooth)
  ES XXX (smooth)
  ES ZZZ (smooth)
  ES FFF (smooth)
  ES SXS (smooth)


## Evaluation Functions

Each package has a different API, so we need separate evaluation functions.

In [6]:
def _evaluate_task(args):
    """
    Worker function for parallel evaluation.
    Handles different package APIs (automatic selection models only).
    """
    import numpy as np
    import time
    import warnings
    warnings.filterwarnings('ignore')
    
    series_idx, series, method_name, package, config = args
    
    try:
        start_time = time.time()
        forecast_values = None
        
        # ===== SMOOTH PACKAGE =====
        if package == "smooth":
            from smooth import ADAM, ES
            
            model_class = config.get("class", "ES")
            model_str = config.get("model", "ZXZ")
            initial = config.get("initial", "backcasting")
            
            if series.period > 1:
                lags = [1, series.period]
            else:
                lags = [1]
                if len(model_str) == 3 and model_str[2] != 'N':
                    model_str = model_str[:2] + 'N'
            
            ModelClass = ADAM if model_class == "ADAM" else ES
            model = ModelClass(model=model_str, lags=lags, initial=initial)
            model.fit(series.x)
            forecasts = model.predict(h=series.h)
            forecast_values = forecasts['mean'].values
        
        # ===== STATSFORECAST =====
        elif package == "statsforecast":
            from statsforecast import models as sf_models
            
            season_length = series.period if series.period > 1 else 1
            model = sf_models.AutoETS(season_length=season_length)
            model.fit(series.x)
            forecasts = model.predict(h=series.h)
            forecast_values = forecasts['mean'] if isinstance(forecasts, dict) else forecasts
        
        # ===== SKTIME =====
        elif package == "sktime":
            import pandas as pd
            from sktime.forecasting import ets as sktime_ets

            y = pd.Series(series.x)
            fh = np.arange(1, series.h + 1)
            sp = series.period if series.period > 1 else 1
            model = sktime_ets.AutoETS(sp=sp, auto=True)
            model.fit(y)
            forecast_values = model.predict(fh).values

        # ===== SKFORECAST =====
        elif package == "skforecast":
            from skforecast.stats import Ets

            season_length = series.period if series.period > 1 else 1
            model = Ets(model="ZZZ", m=season_length)
            model.fit(series.x)
            forecast_values = model.predict(steps=series.h)

        # ===== AEON =====
        elif package == "aeon":
            from aeon.forecasting.stats import AutoETS

            season_length = series.period if series.period > 1 else 1
            model = AutoETS(seasonal_period=season_length)
            x = np.asarray(series.x, dtype=float)
            model.fit(x)
            forecast_values = model.iterative_forecast(x, prediction_horizon=series.h)

        else:
            raise ValueError(f"Unknown package: {package}")
        
        time_elapsed = time.time() - start_time
        
        forecast_values = np.asarray(forecast_values).flatten()[:series.h]
        
        # Calculate metrics
        mse = np.mean((series.xx - forecast_values) ** 2)
        scale = np.mean(np.diff(series.x) ** 2)
        rmsse = np.sqrt(mse / scale) if scale != 0 else np.nan
        
        ame = np.abs(np.mean(series.xx - forecast_values))
        scale_same = np.mean(np.abs(np.diff(series.x)))
        same = ame / scale_same if scale_same != 0 else np.nan
        
        return (series_idx, method_name, rmsse, same, time_elapsed)
    
    except Exception as e:
        return (series_idx, method_name, np.nan, np.nan, np.nan)


def evaluate_parallel(datasets, methods_config, n_workers=None):
    """
    Evaluate all methods on all datasets in parallel.
    """
    if n_workers is None:
        n_workers = multiprocessing.cpu_count()
    
    methods_names = [m[0] for m in methods_config]
    n_methods = len(methods_names)
    n_datasets = len(datasets)
    
    results = np.full((n_methods, n_datasets, 3), np.nan)
    
    tasks = []
    for method_name, package, config in methods_config:
        for i, series in enumerate(datasets):
            tasks.append((i, series, method_name, package, config))
    
    print(f"Starting parallel evaluation with {n_workers} workers...")
    print(f"Total tasks: {len(tasks)} ({n_methods} methods × {n_datasets} series)")
    
    start_time = time.time()
    completed = 0
    
    _ctx = multiprocessing.get_context("fork")
    with ProcessPoolExecutor(max_workers=n_workers, mp_context=_ctx) as executor:
        futures = {executor.submit(_evaluate_task, task): task for task in tasks}
        
        for future in as_completed(futures):
            result = future.result()
            series_idx, method_name, rmsse, same, elapsed = result
            
            method_idx = methods_names.index(method_name)
            results[method_idx, series_idx, 0] = rmsse
            results[method_idx, series_idx, 1] = same
            results[method_idx, series_idx, 2] = elapsed
            
            completed += 1
            if completed % 1000 == 0:
                elapsed_total = time.time() - start_time
                rate = completed / elapsed_total
                remaining = (len(tasks) - completed) / rate
                print(f"  Progress: {completed}/{len(tasks)} ({100*completed/len(tasks):.1f}%) - "
                      f"ETA: {remaining/60:.1f} min")
    
    total_time = time.time() - start_time
    print(f"\nCompleted in {total_time/60:.1f} minutes ({total_time:.1f}s)")
    print(f"Average time per task: {total_time/len(tasks)*1000:.1f}ms")
    
    return results

## Run Evaluation

In [7]:
print(f"Available CPU cores: {multiprocessing.cpu_count()}")

# Run parallel evaluation
import os
_ncap = int(os.environ.get('BENCH_WORKERS', min(30, multiprocessing.cpu_count())))
test_results = evaluate_parallel(datasets, methods_config, n_workers=_ncap)

# Print summary
print("\nPer-method summary:")
for j, method in enumerate(methods_names):
    rmsse_mean = np.nanmean(test_results[j, :, 0])
    same_mean = np.nanmean(test_results[j, :, 1])
    time_mean = np.nanmean(test_results[j, :, 2])
    failed = np.sum(np.isnan(test_results[j, :, 0]))
    print(f"  {method:24s}: RMSSE={rmsse_mean:.4f}, SAME={same_mean:.4f}, "
          f"Time={time_mean:.3f}s, Failed={failed}")

# Save results
import datetime
date_str = datetime.datetime.now().strftime("%Y-%m-%d")
np.save(f'{date_str}-benchmark-ets-all.npy', test_results)

Available CPU cores: 32
Starting parallel evaluation with 30 workers...
Total tasks: 53150 (10 methods × 5315 series)


  Progress: 1000/53150 (1.9%) - ETA: 5.2 min


  Progress: 2000/53150 (3.8%) - ETA: 3.4 min


  Progress: 3000/53150 (5.6%) - ETA: 3.7 min


  Progress: 4000/53150 (7.5%) - ETA: 4.7 min


...

  Progress: 50000/53150 (94.1%) - ETA: 1.5 min


  Progress: 51000/53150 (96.0%) - ETA: 1.0 min


  Progress: 52000/53150 (97.8%) - ETA: 0.6 min


  Progress: 53000/53150 (99.7%) - ETA: 0.1 min



Completed in 26.6 minutes (1596.0s)
Average time per task: 30.0ms

Per-method summary:
  ADAM ETS Back           : RMSSE=1.9530, SAME=1.9817, Time=0.412s, Failed=0
  ADAM ETS Opt            : RMSSE=1.9494, SAME=1.9696, Time=0.507s, Failed=0
  ADAM ETS Two            : RMSSE=1.9607, SAME=1.9902, Time=1.481s, Failed=0
  ES Back                 : RMSSE=1.9487, SAME=1.9735, Time=0.411s, Failed=0
  ES Opt                  : RMSSE=1.9481, SAME=1.9680, Time=0.493s, Failed=0
  ES Two                  : RMSSE=1.9574, SAME=1.9866, Time=1.481s, Failed=0
  ES XXX                  : RMSSE=1.9796, SAME=2.0161, Time=0.215s, Failed=0
  ES ZZZ                  : RMSSE=2.0543, SAME=2.1101, Time=0.574s, Failed=0
  ES FFF                  : RMSSE=2.0997, SAME=2.1737, Time=2.484s, Failed=0
  ES SXS                  : RMSSE=1.9619, SAME=1.9889, Time=0.935s, Failed=0


## Results Summary

In [8]:
import datetime
date_str = datetime.datetime.now().strftime('%Y-%m-%d')
test_results = np.load(f'{date_str}-benchmark-ets-all.npy')

# Create RMSSE summary DataFrame (all series)
summary_rmsse = pd.DataFrame({
    'Method': methods_names,
    'Min': [np.nanmin(test_results[j, :, 0]) for j in range(methods_number)],
    'Q1': [np.nanquantile(test_results[j, :, 0], 0.25) for j in range(methods_number)],
    'Med': [np.nanmedian(test_results[j, :, 0]) for j in range(methods_number)],
    'Q3': [np.nanquantile(test_results[j, :, 0], 0.75) for j in range(methods_number)],
    'Max': [np.nanmax(test_results[j, :, 0]) for j in range(methods_number)],
    'Mean': [np.nanmean(test_results[j, :, 0]) for j in range(methods_number)],
    'Mean Time (s)': [np.nanmean(test_results[j, :, 2]) for j in range(methods_number)],
    'Failed': [np.sum(np.isnan(test_results[j, :, 0])) for j in range(methods_number)]
})

print("\n" + "="*85)
print("EVALUATION RESULTS for RMSSE (All Series)")
print("="*85)
print(summary_rmsse.to_string(index=False))


EVALUATION RESULTS for RMSSE (All Series)
       Method      Min       Q1      Med       Q3        Max     Mean  Mean Time (s)  Failed
ADAM ETS Back 0.018252 0.671249 1.187745 2.335668  51.615990 1.953038       0.412099       0
 ADAM ETS Opt 0.022098 0.668100 1.182516 2.341091  51.615990 1.949359       0.507011       0
 ADAM ETS Two 0.024599 0.669689 1.195231 2.368713  51.615990 1.960704       1.481336       0
      ES Back 0.018252 0.674360 1.189752 2.312608  51.615990 1.948698       0.410554       0
       ES Opt 0.022098 0.670717 1.191364 2.360373  51.615990 1.948147       0.492601       0
       ES Two 0.024467 0.677858 1.196102 2.347646  51.615990 1.957357       1.480782       0
       ES XXX 0.018252 0.685003 1.183571 2.326863  51.615990 1.979638       0.215483       0
       ES ZZZ 0.011386 0.674360 1.196847 2.334080 115.495387 2.054267       0.574444       0
       ES FFF 0.011386 0.688002 1.222222 2.446776  58.610410 2.099736       2.484024       0
       ES SXS 0.018252 0.67

In [9]:
# Create SAME summary DataFrame (all series)
summary_same = pd.DataFrame({
    'Method': methods_names,
    'Min': [np.nanmin(test_results[j, :, 1]) for j in range(methods_number)],
    'Q1': [np.nanquantile(test_results[j, :, 1], 0.25) for j in range(methods_number)],
    'Med': [np.nanmedian(test_results[j, :, 1]) for j in range(methods_number)],
    'Q3': [np.nanquantile(test_results[j, :, 1], 0.75) for j in range(methods_number)],
    'Max': [np.nanmax(test_results[j, :, 1]) for j in range(methods_number)],
    'Mean': [np.nanmean(test_results[j, :, 1]) for j in range(methods_number)],
    'Mean Time (s)': [np.nanmean(test_results[j, :, 2]) for j in range(methods_number)],
    'Failed': [np.sum(np.isnan(test_results[j, :, 1])) for j in range(methods_number)]
})

print("\n" + "="*85)
print("EVALUATION RESULTS for SAME (All Series)")
print("="*85)
print(summary_same.to_string(index=False))


EVALUATION RESULTS for SAME (All Series)
       Method      Min       Q1      Med       Q3        Max     Mean  Mean Time (s)  Failed
ADAM ETS Back 0.000261 0.378749 1.018707 2.435576  56.281980 1.981727       0.412099       0
 ADAM ETS Opt 0.000148 0.380206 1.014527 2.462395  55.101789 1.969558       0.507011       0
 ADAM ETS Two 0.000072 0.388245 1.036665 2.475049  55.101861 1.990236       1.481336       0
      ES Back 0.000041 0.385218 1.033891 2.422379  56.517594 1.973493       0.410554       0
       ES Opt 0.000136 0.374619 1.012840 2.449642  54.686030 1.968047       0.492601       0
       ES Two 0.000519 0.393084 1.043207 2.486045  54.685580 1.986586       1.480782       0
       ES XXX 0.000361 0.385912 1.029440 2.422823  54.428407 2.016097       0.215483       0
       ES ZZZ 0.000041 0.397402 1.043831 2.470550 145.700451 2.110125       0.574444       0
       ES FFF 0.000041 0.409465 1.062759 2.573658  60.336377 2.173661       2.484024       0
       ES SXS 0.000041 0.387

In [10]:
# Create Time summary DataFrame (all series)
summary_time = pd.DataFrame({
    'Method': methods_names,
    'Min': [np.nanmin(test_results[j, :, 2]) for j in range(methods_number)],
    'Q1': [np.nanquantile(test_results[j, :, 2], 0.25) for j in range(methods_number)],
    'Med': [np.nanmedian(test_results[j, :, 2]) for j in range(methods_number)],
    'Q3': [np.nanquantile(test_results[j, :, 2], 0.75) for j in range(methods_number)],
    'Max': [np.nanmax(test_results[j, :, 2]) for j in range(methods_number)],
    'Mean': [np.nanmean(test_results[j, :, 2]) for j in range(methods_number)],
    'Total (min)': [np.nansum(test_results[j, :, 2]) / 60 for j in range(methods_number)],
    'Failed': [np.sum(np.isnan(test_results[j, :, 2])) for j in range(methods_number)]
})

print("\n" + "="*85)
print("EVALUATION RESULTS for Computational Time in seconds (All Series)")
print("="*85)
print(summary_time.to_string(index=False))


EVALUATION RESULTS for Computational Time in seconds (All Series)
       Method      Min       Q1      Med       Q3       Max     Mean  Total (min)  Failed
ADAM ETS Back 0.006109 0.041551 0.100818 0.269358 10.917419 0.412099    36.505110       0
 ADAM ETS Opt 0.034917 0.201628 0.294901 0.747337  2.333542 0.507011    44.912746       0
 ADAM ETS Two 0.035640 0.247667 0.430492 1.263822 25.946972 1.481336   131.221661       0
      ES Back 0.006080 0.042647 0.100858 0.289721  9.676007 0.410554    36.368205       0
       ES Opt 0.030468 0.197476 0.289707 0.724757  2.253837 0.492601    43.636269       0
       ES Two 0.036485 0.239859 0.418750 1.231209 25.724215 1.480782   131.172602       0
       ES XXX 0.005044 0.031096 0.065459 0.168668  4.812356 0.215483    19.088220       0
       ES ZZZ 0.006832 0.045936 0.159642 0.382496 14.182687 0.574444    50.886158       0
       ES FFF 0.028146 0.196624 0.910792 2.359550 32.988491 2.484024   220.043121       0
       ES SXS 0.014236 0.104110 0

## Series failed diagnostics

In [11]:
# Build series ID list matching the order in `datasets`
series_ids = list(M1.keys()) + list(M3.keys()) + list(Tourism.keys())
dataset_sizes = {"M1": len(M1), "M3": len(M3), "Tourism": len(Tourism)}
dataset_boundaries = {}
offset = 0
for name, size in dataset_sizes.items():
    dataset_boundaries[name] = (offset, offset + size)
    offset += size

print("=" * 70)
print("FAILED SERIES DIAGNOSTICS")
print("=" * 70)

any_failures = False
for j, method in enumerate(methods_names):
    failed_mask = np.isnan(test_results[j, :, 0])
    n_failed = failed_mask.sum()
    if n_failed == 0:
        continue
    any_failures = True
    failed_indices = np.where(failed_mask)[0]

    print(f"\n{method} ({n_failed} failures):")
    for ds_name, (lo, hi) in dataset_boundaries.items():
        ds_failed = [series_ids[i] for i in failed_indices if lo <= i < hi]
        if ds_failed:
            print(f"  {ds_name}: {', '.join(str(s) for s in ds_failed)}")

if not any_failures:
    print("\nNo failures detected.")

FAILED SERIES DIAGNOSTICS

No failures detected.
